    "# Fisher information & aPRF session shifts\n",
    "\n",
    "Combined analysis:\n",
    "1. **Von Mises** → gabor orientation FI (BensonV1)\n",
    "2. **aPRF standard** → abstract value FI (NPCr)\n",
    "3. **aPRF session-shift** → value FI per session + mode shift analysis (NPCr)"

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn.maskers import NiftiMasker

from abstract_values.utils.data import Subject, BIDS_FOLDER

sns.set_theme(style='ticks', font_scale=1.1)

In [ ]:
N_VOXELS = 100
SMOOTHED = False
R2_THR   = 0.05   # for shift analysis voxel selection

MAPPING_PALETTE = {'cdf': 'steelblue', 'inverse_cdf': 'tomato'}

# Value distribution peaks in orientation space (for Von Mises FI)
MAPPING_ORI_PEAKS = {'cdf': [45.0, 135.0], 'inverse_cdf': [90.0]}

# Value distribution peaks in CHF space (for aPRF FI)
MAPPING_VALUE_PEAKS = {'cdf': [12.0, 32.0], 'inverse_cdf': [22.0]}

VALUE_MIN, VALUE_MAX = 2.5, 41.5

# Auto-discover subjects
SUBJECTS = sorted({
    d.name.removeprefix('sub-')
    for model_dir in ['aprf', 'aprf-session-shift', 'vonmises']
    if (BIDS_FOLDER / 'derivatives' / 'encoding_models' / model_dir).exists()
    for d in (BIDS_FOLDER / 'derivatives' / 'encoding_models' / model_dir).iterdir()
    if d.is_dir() and d.name.startswith('sub-')
})
print(f'Subjects: {SUBJECTS}')

In [ ]:
def get_mapping(subject, session):
    """Return 'cdf' or 'inverse_cdf'."""
    num = int(''.join(c for c in str(subject) if c.isdigit()))
    if num % 2 == 0:
        return 'cdf' if session == 1 else 'inverse_cdf'
    return 'inverse_cdf' if session == 1 else 'cdf'


def _find_fi_file(sub_dir, subject, session, mask_desc,
                  n_voxels_list, smoothed):
    """Try to find a FI file, searching n_voxels variants and session/no-session paths."""
    smooth_label = '_smoothed' if smoothed else ''

    for nv in (n_voxels_list if isinstance(n_voxels_list, list) else [n_voxels_list]):
        # Try per-session path
        if session is not None:
            fn = (sub_dir / f'ses-{session}' / 'func' /
                  f'sub-{subject}_ses-{session}_task-abstractvalue'
                  f'_mask-{mask_desc}_nvoxels-{nv}{smooth_label}_desc-fisherinfo_pe.tsv')
            if fn.exists():
                return fn, nv

        # Try all-sessions path
        fn = (sub_dir / 'func' /
              f'sub-{subject}_task-abstractvalue'
              f'_mask-{mask_desc}_nvoxels-{nv}{smooth_label}_desc-fisherinfo_pe.tsv')
        if fn.exists():
            return fn, nv

    return None, None


def load_fi_vonmises(subject, session=None, roi='BensonV1', hemi='LR',
                     n_voxels=100, smoothed=False):
    """Load Von Mises (gabor) Fisher information."""
    hemi_str = f'_hemi-{hemi}' if hemi else ''
    mask_desc = f'{roi}{hemi_str}'
    sub_dir = (BIDS_FOLDER / 'derivatives' / 'encoding_models' / 'vonmises'
               / f'sub-{subject}')

    fn, nv_used = _find_fi_file(sub_dir, subject, session, mask_desc,
                                [n_voxels, 250, 100, 500], smoothed)
    if fn is None:
        return None
    df = pd.read_csv(fn, sep='\t', index_col=0)
    df.index = np.rad2deg(df.index)
    df.index.name = 'orientation_deg'
    return df


def load_fi_aprf(subject, session=None, model='aprf', roi='NPCr', hemi=None,
                 n_voxels=100, smoothed=False):
    """Load aPRF (value) Fisher information."""
    mask_desc = roi
    sub_dir = (BIDS_FOLDER / 'derivatives' / 'encoding_models' / model
               / f'sub-{subject}')

    fn, nv_used = _find_fi_file(sub_dir, subject, session, mask_desc,
                                [n_voxels, 250, 100], smoothed)
    if fn is None:
        return None
    df = pd.read_csv(fn, sep='\t', index_col=0)
    df.index.name = 'value_chf'
    return df

---
# 1. Gabor orientation Fisher information (Von Mises)

In [ ]:
vm_records = []

for subject in SUBJECTS:
    sub = Subject(subject, bids_folder=BIDS_FOLDER)
    try:
        sessions = sub.get_sessions()
    except FileNotFoundError:
        continue

    # Try per-session files first
    found_per_session = False
    for ses in sessions:
        fi = load_fi_vonmises(subject, session=ses, n_voxels=N_VOXELS, smoothed=SMOOTHED)
        if fi is None:
            continue
        found_per_session = True
        mapping = get_mapping(subject, ses)
        ori = fi.index.values
        log_fi = np.log(fi.iloc[:, 0].values)
        for o, lf in zip(ori, log_fi):
            vm_records.append(dict(
                subject=subject, session=ses, mapping=mapping,
                orientation=o, log_fi=lf))

    # If no per-session files, load all-sessions file once
    if not found_per_session:
        fi = load_fi_vonmises(subject, session=None, n_voxels=N_VOXELS, smoothed=SMOOTHED)
        if fi is not None:
            ori = fi.index.values
            log_fi = np.log(fi.iloc[:, 0].values)
            for o, lf in zip(ori, log_fi):
                vm_records.append(dict(
                    subject=subject, session='all', mapping='pooled',
                    orientation=o, log_fi=lf))

vm_df = pd.DataFrame(vm_records)
if not vm_df.empty:
    print(f'{vm_df["subject"].nunique()} subjects, {len(vm_df)} rows')
else:
    print('No Von Mises FI data.')

In [ ]:
if not vm_df.empty:
    subjects = sorted(vm_df['subject'].unique())
    fig, axes = plt.subplots(1, len(subjects),
                              figsize=(5 * len(subjects), 4),
                              sharey=False, constrained_layout=True)
    if len(subjects) == 1:
        axes = [axes]

    for ax, sub in zip(axes, subjects):
        sub_df = vm_df[vm_df['subject'] == sub]
        seen = set()
        for ses in sorted(sub_df['session'].unique(), key=str):
            ses_df = sub_df[sub_df['session'] == ses]
            mapping = ses_df['mapping'].iloc[0]
            if mapping == 'pooled':
                color = '0.3'
                label_str = 'all sessions (pooled)'
            else:
                color = MAPPING_PALETTE[mapping]
                label_str = f'ses-{ses} ({mapping})'
            ori = ses_df['orientation'].values
            lfi = ses_df['log_fi'].values
            peak = ori[lfi.argmax()]
            ax.plot(ori, lfi, lw=2, color=color,
                    label=f'{label_str}  peak={peak:.0f}°')
            ax.axvline(peak, color=color, lw=1, ls='--', alpha=0.5)
            if mapping not in seen and mapping != 'pooled':
                for dp in MAPPING_ORI_PEAKS[mapping]:
                    ax.axvline(dp, color=color, lw=1, ls=':', alpha=0.3)
                seen.add(mapping)

        ax.set_xlabel('Orientation (°)')
        ax.set_ylabel('log Fisher information')
        ax.set_title(f'sub-{sub}')
        ax.legend(fontsize=8)

    fig.suptitle(f'Gabor orientation FI (Von Mises, BensonV1 LR, top-{N_VOXELS})\n'
                 'Dotted: value-distribution peaks in orientation space',
                 fontsize=11)
    plt.show()

---
# 2. Abstract value Fisher information (standard aPRF)

In [ ]:
aprf_records = []

for subject in SUBJECTS:
    sub = Subject(subject, bids_folder=BIDS_FOLDER)
    try:
        sessions = sub.get_sessions()
    except FileNotFoundError:
        continue

    # Try per-session files first
    found_per_session = False
    for ses in sessions:
        fi = load_fi_aprf(subject, session=ses, model='aprf',
                          n_voxels=N_VOXELS, smoothed=SMOOTHED)
        if fi is None:
            continue
        found_per_session = True
        mapping = get_mapping(subject, ses)
        vals = fi.index.values.astype(float)
        log_fi = np.log(np.clip(fi.iloc[:, 0].values, 1e-10, None))
        for v, lf in zip(vals, log_fi):
            aprf_records.append(dict(
                subject=subject, session=ses, mapping=mapping,
                model='standard', value_chf=v, log_fi=lf))

    # If no per-session files, load all-sessions file once
    if not found_per_session:
        fi = load_fi_aprf(subject, session=None, model='aprf',
                          n_voxels=N_VOXELS, smoothed=SMOOTHED)
        if fi is not None:
            vals = fi.index.values.astype(float)
            log_fi = np.log(np.clip(fi.iloc[:, 0].values, 1e-10, None))
            for v, lf in zip(vals, log_fi):
                aprf_records.append(dict(
                    subject=subject, session='all', mapping='pooled',
                    model='standard', value_chf=v, log_fi=lf))

aprf_df = pd.DataFrame(aprf_records)
if not aprf_df.empty:
    print(f'{aprf_df["subject"].nunique()} subjects, {len(aprf_df)} rows')
else:
    print('No standard aPRF FI data.')

In [ ]:
if not aprf_df.empty:
    subjects = sorted(aprf_df['subject'].unique())
    fig, axes = plt.subplots(1, len(subjects),
                              figsize=(5 * len(subjects), 4),
                              sharey=False, constrained_layout=True)
    if len(subjects) == 1:
        axes = [axes]

    for ax, sub in zip(axes, subjects):
        sub_df = aprf_df[aprf_df['subject'] == sub]
        for ses in sorted(sub_df['session'].unique(), key=str):
            ses_df = sub_df[sub_df['session'] == ses]
            mapping = ses_df['mapping'].iloc[0]
            if mapping == 'pooled':
                color = '0.3'
                label_str = 'all sessions (pooled)'
            else:
                color = MAPPING_PALETTE[mapping]
                label_str = f'ses-{ses} ({mapping})'
            vals = ses_df['value_chf'].values
            lfi = ses_df['log_fi'].values
            peak = vals[lfi.argmax()]
            ax.plot(vals, lfi, lw=2, color=color,
                    label=f'{label_str}  peak={peak:.1f} CHF')
            ax.axvline(peak, color=color, lw=1, ls='--', alpha=0.5)

        ax.set_xlabel('Value (CHF)')
        ax.set_ylabel('log Fisher information')
        ax.set_title(f'sub-{sub}')
        ax.legend(fontsize=8)

    fig.suptitle(f'Value FI (standard aPRF, NPCr, top-{N_VOXELS})',
                 fontsize=11)
    plt.show()

---
# 3. Abstract value Fisher information (session-shift aPRF)

In the session-shift model, the mode of each voxel's tuning curve shifts freely
between sessions while fwhm/amplitude/baseline are shared. This produces
session-specific FI curves that reflect how value representations shift with
the learned mapping.

In [ ]:
shift_records = []

for subject in SUBJECTS:
    sub = Subject(subject, bids_folder=BIDS_FOLDER)
    try:
        sessions = sub.get_sessions()
    except FileNotFoundError:
        continue

    for ses in sessions:
        fi = load_fi_aprf(subject, session=ses, model='aprf-session-shift',
                          n_voxels=N_VOXELS, smoothed=SMOOTHED)
        if fi is None:
            continue
        # Skip all-zero FI (degenerate fits)
        if (fi.iloc[:, 0] == 0).all():
            print(f'  sub-{subject} ses-{ses}: all-zero shift FI, skipping')
            continue
        mapping = get_mapping(subject, ses)
        vals = fi.index.values.astype(float)
        log_fi = np.log(np.clip(fi.iloc[:, 0].values, 1e-10, None))
        for v, lf in zip(vals, log_fi):
            shift_records.append(dict(
                subject=subject, session=ses, mapping=mapping,
                model='session-shift',
                value_chf=v, log_fi=lf))

shift_df = pd.DataFrame(shift_records)
if not shift_df.empty:
    print(f'{shift_df["subject"].nunique()} subjects, {len(shift_df)} rows')
else:
    print('No session-shift aPRF FI data.')

In [ ]:
if not shift_df.empty:
    subjects = sorted(shift_df['subject'].unique())
    fig, axes = plt.subplots(1, len(subjects),
                              figsize=(5 * len(subjects), 4),
                              sharey=False, constrained_layout=True)
    if len(subjects) == 1:
        axes = [axes]

    for ax, sub in zip(axes, subjects):
        sub_df = shift_df[shift_df['subject'] == sub]
        for ses in sorted(sub_df['session'].unique()):
            ses_df = sub_df[sub_df['session'] == ses]
            mapping = ses_df['mapping'].iloc[0]
            color = MAPPING_PALETTE[mapping]
            vals = ses_df['value_chf'].values
            lfi = ses_df['log_fi'].values
            peak = vals[lfi.argmax()]
            ax.plot(vals, lfi, lw=2, color=color,
                    label=f'ses-{ses} ({mapping})  peak={peak:.1f} CHF')
            ax.axvline(peak, color=color, lw=1, ls='--', alpha=0.5)

        ax.set_xlabel('Value (CHF)')
        ax.set_ylabel('log Fisher information')
        ax.set_title(f'sub-{sub}')
        ax.legend(fontsize=8)

    fig.suptitle(f'Value FI (session-shift aPRF, NPCr, top-{N_VOXELS})',
                 fontsize=11)
    plt.show()

---
# 4. Comparison: standard vs session-shift aPRF

In [ ]:
both = pd.concat([aprf_df, shift_df], ignore_index=True)

if not both.empty and both['model'].nunique() > 1:
    subjects = sorted(both['subject'].unique())
    sessions_all = sorted(both['session'].unique())

    fig, axes = plt.subplots(len(subjects), len(sessions_all),
                              figsize=(5 * len(sessions_all), 4 * len(subjects)),
                              squeeze=False, constrained_layout=True)

    model_styles = {'standard': '-', 'session-shift': '--'}

    for i, sub in enumerate(subjects):
        for j, ses in enumerate(sessions_all):
            ax = axes[i, j]
            sub_ses = both[(both['subject'] == sub) & (both['session'] == ses)]
            if sub_ses.empty:
                ax.set_visible(False)
                continue
            mapping = sub_ses['mapping'].iloc[0]
            color = MAPPING_PALETTE[mapping]

            for model, ls in model_styles.items():
                m_df = sub_ses[sub_ses['model'] == model]
                if m_df.empty:
                    continue
                ax.plot(m_df['value_chf'], m_df['log_fi'],
                        lw=2, ls=ls, color=color, label=model)

            ax.set_xlabel('Value (CHF)')
            ax.set_ylabel('log FI')
            ax.set_title(f'sub-{sub}  ses-{ses} ({mapping})')
            ax.legend(fontsize=8)

    fig.suptitle(f'Standard vs session-shift aPRF  (NPCr, top-{N_VOXELS})',
                 fontsize=12)
    plt.show()
else:
    print('Need both standard and session-shift data for comparison.')

---
# 5. aPRF mode shifts between sessions

Load session-shift model parameters (mode_1, mode_2) within NPCr and visualize
how preferred values shift between the two mapping conditions.

In [ ]:
# ── load session-shift mode parameters ────────────────────────────────────
shift_data = {}

for subject in SUBJECTS:
    sub = Subject(subject, bids_folder=BIDS_FOLDER)
    out_dir = (BIDS_FOLDER / 'derivatives' / 'encoding_models'
               / 'aprf-session-shift' / f'sub-{subject}' / 'func')

    def _path(desc):
        return out_dir / f'sub-{subject}_task-abstractvalue_space-T1w_desc-{desc}_pe.nii.gz'

    if not _path('r2').exists():
        print(f'sub-{subject}: no session-shift model, skipping')
        continue

    try:
        mask = sub.get_roi_mask('NPCr', hemi=None)
    except FileNotFoundError:
        print(f'sub-{subject}: no NPCr mask, skipping')
        continue

    masker = NiftiMasker(mask_img=mask).fit()
    r2    = masker.transform(_path('r2')).squeeze()
    mode1 = masker.transform(_path('mode_1')).squeeze()
    mode2 = masker.transform(_path('mode_2')).squeeze()

    sel = r2 > R2_THR
    print(f'sub-{subject}: {sel.sum():,} / {len(r2):,} voxels above R²={R2_THR}')

    shift_data[subject] = dict(
        r2=r2, mode1=mode1, mode2=mode2, sel=sel,
        map1=get_mapping(subject, 1),
        map2=get_mapping(subject, 2),
    )

In [ ]:
# ── joint mode plot (hexbin) ───────────────────────────────────────────────
if shift_data:
    subjects = sorted(shift_data.keys())
    fig, axes = plt.subplots(1, len(subjects),
                              figsize=(6 * len(subjects), 5.5),
                              squeeze=False, constrained_layout=True)

    for ax, subject in zip(axes[0], subjects):
        d = shift_data[subject]
        m1, m2 = d['mode1'][d['sel']], d['mode2'][d['sel']]
        map1, map2 = d['map1'], d['map2']

        hb = ax.hexbin(m1, m2, gridsize=40, cmap='viridis',
                       extent=[VALUE_MIN, VALUE_MAX, VALUE_MIN, VALUE_MAX],
                       mincnt=1)
        ax.plot([VALUE_MIN, VALUE_MAX], [VALUE_MIN, VALUE_MAX],
                'w--', lw=1, alpha=0.5)

        for px in MAPPING_VALUE_PEAKS[map1]:
            ax.axvline(px, color='cyan', lw=1.5, ls=':', alpha=0.8)
        for py in MAPPING_VALUE_PEAKS[map2]:
            ax.axhline(py, color='orange', lw=1.5, ls='--', alpha=0.8)

        ax.set_xlabel(f'Mode ses-1 ({map1}) [CHF]')
        ax.set_ylabel(f'Mode ses-2 ({map2}) [CHF]')
        ax.set_title(f'sub-{subject}  (R²>{R2_THR}, n={d["sel"].sum():,})')
        ax.set_xlim(VALUE_MIN, VALUE_MAX)
        ax.set_ylim(VALUE_MIN, VALUE_MAX)
        ax.set_aspect('equal')
        plt.colorbar(hb, ax=ax, label='voxel count', shrink=0.8)

    fig.suptitle('Session-shift aPRF: preferred value per session (NPCr)',
                 fontsize=12)
    plt.show()
else:
    print('No session-shift data available.')

In [ ]:
# ── delta-mode histograms ──────────────────────────────────────────────────
if shift_data:
    subjects = sorted(shift_data.keys())
    fig, axes = plt.subplots(1, len(subjects),
                              figsize=(6 * len(subjects), 4),
                              squeeze=False, constrained_layout=True)

    for ax, subject in zip(axes[0], subjects):
        d = shift_data[subject]
        m1, m2 = d['mode1'][d['sel']], d['mode2'][d['sel']]
        delta = m2 - m1
        map1, map2 = d['map1'], d['map2']

        ax.hist(delta, bins=60, color='steelblue', edgecolor='none')
        ax.axvline(0, color='k', lw=1.5, ls='--')
        ax.axvline(delta.mean(), color='tomato', lw=1.5,
                   label=f'mean = {delta.mean():.2f} CHF')

        ax.set_xlabel(f'mode ses-2 ({map2}) - mode ses-1 ({map1}) [CHF]')
        ax.set_ylabel('voxel count')
        ax.set_title(f'sub-{subject}')
        ax.legend()

    fig.suptitle('Shift in preferred value between sessions (NPCr)', fontsize=12)
    plt.show()